# DG-ALETSK: reproducing the article step by step

This notebook reproduces part of Table V of

> G. Xue, J. Wang, B. Yuan and C. Dai, "DG-ALETSK: a high-dimensional fuzzy approach with
> simultaneous feature selection and rule extraction", *IEEE Transactions on Fuzzy
> Systems*, vol. 31, no. 11, 2023.

It runs the same code as the script `examples/reproductions/dgaletsk_2023.py`, one step
at a time:

1. settings and reproducibility;
2. the data;
3. one fold in detail;
4. the full protocol of the article, ten-fold cross-validation repeated five times;
5. the comparison with the article.

The full protocol takes about fifteen minutes on a laptop, on four threads.

## 1. Settings and reproducibility

In [1]:
import numpy as np
import torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler

import highfis
from highfis import DGALETSKClassifier

The seed, the numbers of repetitions and folds, and the number of threads are fixed here.
The values of the article are copied from its Table V.

In [2]:
SEED = 0
REPETITIONS = 5
FOLDS = 10
THREADS = 4

# OpenML identifier of each dataset.
DATASETS = {"Colon": 1432}

# Table V of the article: accuracy in percent, selected features, extracted rules.
ARTICLE = {"Colon": (81.95, 6.80, 6.42)}

Three lines make the run repeatable: a fixed number of threads, deterministic algorithms
in PyTorch, and no reuse of fuzzy sets between fits.

In [3]:
torch.set_num_threads(THREADS)
torch.use_deterministic_algorithms(True)
highfis.set_mf_cache_enabled(False)

## 2. The data

Colon is a gene-expression dataset: 62 tissue samples described by 2000 genes, in two
classes. OpenML provides it with the samples and features of the article. It is stored as
a sparse matrix, which is converted to a dense one.

In [4]:
bunch = fetch_openml(data_id=DATASETS["Colon"], as_frame=False, parser="liac-arff")
data = bunch.data.toarray() if hasattr(bunch.data, "toarray") else bunch.data
features = np.asarray(data, dtype=np.float32)
labels = np.unique(bunch.target, return_inverse=True)[1]

print(f"Colon: {features.shape[0]} samples, {features.shape[1]} features, {labels.max() + 1} classes")
print(f"samples per class: {np.bincount(labels).tolist()}")

Colon: 62 samples, 2000 features, 2 classes
samples per class: [40, 22]


## 3. One fold in detail

Before running the whole protocol, this section follows the first fold.

### The split and the scaling

The article scales the inputs to [0, 1]. The range is taken from the training part only.
With 62 samples, a test part has six or seven of them: one sample is about 15 points of
the accuracy of a fold.

In [5]:
folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
train, test = next(folds.split(features, labels))

scaler = MinMaxScaler().fit(features[train])
x_train, x_test = scaler.transform(features[train]), scaler.transform(features[test])

print(f"training part: {x_train.shape[0]} samples, test part: {x_test.shape[0]} samples")

training part: 55 samples, test part: 7 samples


### The model

DG-ALETSK builds a rule base with one rule per training sample, computes the firing
strengths with an adaptive softmin, and trains two groups of gates, one per feature and
one per rule. The gates are trained for 10 epochs; the features and rules whose gates
stay below a threshold are removed, and the remaining system is fine-tuned.

These are the defaults of `DGALETSKClassifier`, so the model is built without arguments
other than the seed.

In [6]:
model = DGALETSKClassifier(random_state=SEED).fit(x_train, labels[train])

print(f"test accuracy: {100.0 * model.score(x_test, labels[test]):.2f}%")
print(f"selected features: {len(model.selected_features_)} of {x_train.shape[1]}")
print(f"extracted rules: {model.model_.n_rules} of {x_train.shape[0]}")

test accuracy: 85.71%
selected features: 7 of 2000
extracted rules: 2 of 55


The selected genes, by their column in the data:

In [7]:
print(sorted(int(d) for d in model.selected_features_))

[248, 364, 376, 492, 821, 1422, 1770]


## 4. The full protocol

The article reports the mean over ten-fold cross-validation repeated five times, that is,
50 runs. The loop below repeats the previous section for every run and stores the
accuracy and the numbers of selected features and extracted rules.

In [8]:
results = {name: [] for name in DATASETS}

for name, data_id in DATASETS.items():
    bunch = fetch_openml(data_id=data_id, as_frame=False, parser="liac-arff")
    # OpenML stores some datasets, Colon among them, as sparse matrices.
    data = bunch.data.toarray() if hasattr(bunch.data, "toarray") else bunch.data
    features = np.asarray(data, dtype=np.float32)
    labels = np.unique(bunch.target, return_inverse=True)[1]

    for repetition in range(REPETITIONS):
        folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED + repetition)
        for train, test in folds.split(features, labels):
            scaler = MinMaxScaler().fit(features[train])
            x_train, x_test = scaler.transform(features[train]), scaler.transform(features[test])

            model = DGALETSKClassifier(random_state=SEED + repetition).fit(x_train, labels[train])
            results[name].append(
                (100.0 * model.score(x_test, labels[test]), len(model.selected_features_), model.model_.n_rules)
            )
        print(f"{name}: repetition {repetition + 1} of {REPETITIONS} done")

Colon: repetition 1 of 5 done


Colon: repetition 2 of 5 done


Colon: repetition 3 of 5 done


Colon: repetition 4 of 5 done


Colon: repetition 5 of 5 done


## 5. Comparison with the article

In [9]:
print("Table V: DG-ALETSK (accuracy in percent / selected features / extracted rules)")
print(f"{'dataset':8s} {'article':>20s} {'highFIS':>20s} {'std of accuracy':>16s}")
for name in DATASETS:
    accuracy, n_features, n_rules = np.asarray(results[name]).T
    reported = "{:.2f} / {:.2f} / {:.2f}".format(*ARTICLE[name])
    measured = f"{accuracy.mean():.2f} / {n_features.mean():.2f} / {n_rules.mean():.2f}"
    print(f"{name:8s} {reported:>20s} {measured:>20s} {accuracy.std():16.2f}")

Table V: DG-ALETSK (accuracy in percent / selected features / extracted rules)
dataset               article              highFIS  std of accuracy
Colon     81.95 / 6.80 / 6.42  86.95 / 7.18 / 6.98            10.75


The numbers of selected genes and of extracted rules are those of the article. The
accuracy is five points above it, half the standard deviation between the 50 runs: with
62 samples, one test sample is 1.6 points of a cross-validated accuracy.

How the accuracy of the 50 runs is distributed:

In [10]:
accuracy = np.asarray(results["Colon"])[:, 0]
values, counts = np.unique(np.round(accuracy, 1), return_counts=True)
for value, count in zip(values, counts):
    print(f"{value:6.1f}%  {'#' * count} {count}")

  57.1%  # 1
  66.7%  #### 4
  71.4%  # 1
  83.3%  #################### 20
  85.7%  ######## 8
 100.0%  ################ 16


## Environment

The versions that produced the numbers above:

In [11]:
highfis.show_versions()

highfis       : 0.33.0
python        : 3.11.15
platform      : Linux-7.0.0-38-generic-x86_64-with-glibc2.43
torch         : 2.14.1+cu130
numpy         : 2.4.6
scikit-learn  : 1.9.1
default dtype : torch.float32
threads       : 4
cpu           : Intel(R) Core(TM) Ultra 7 258V
